# 6.19 — Gradient Clipping

Gradient clipping keeps a training step from becoming dangerously large when a raw gradient spikes. The key idea is not to change what direction the gradient points unless we must; we measure its norm, compare that length to a threshold, and rescale only when the length exceeds the safe budget.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build gradient clipping one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so nothing is a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vector norms, and small optimization loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the few stochastic demos.

### 1. A gradient is an update direction with a length

A gradient vector tells an optimizer which way the loss rises fastest. Gradient descent moves in the opposite direction, and the movement size is the learning rate times the gradient length. If the length is ordinary, the step is a controlled nudge; if the length explodes, the same learning rate becomes a launch catapult.

In [ ]:
g_w = np.array([3.0, 4.0])  # a two-parameter gradient with visible 3-4-5 geometry.
eta_w = 0.1  # learning rate used to turn a gradient into a parameter update.
norm_w = np.linalg.norm(g_w)  # sqrt(3^2 + 4^2) = 5.
step_w = -eta_w * g_w  # gradient descent moves opposite the gradient.
print("gradient:", g_w)
print("gradient norm:", norm_w)
print("descent step:", step_w, "step length:", np.linalg.norm(step_w))
assert norm_w == 5.0

▶ What you'll see: the raw gradient has length 5, so a learning rate of 0.1 produces a step of length 0.5.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.quiver([0], [0], [g_w[0]], [g_w[1]], angles="xy", scale_units="xy", scale=1, color="crimson", label="gradient g")
plt.quiver([0], [0], [step_w[0]], [step_w[1]], angles="xy", scale_units="xy", scale=1, color="seagreen", label="descent step -ηg")
plt.xlim(-1, 4); plt.ylim(-1, 5)
plt.xlabel("parameter 0"); plt.ylabel("parameter 1")
plt.title("1: direction and length of a gradient"); plt.legend(); plt.show()

▶ What you'll see: the descent arrow points exactly opposite the gradient arrow and is shorter by the learning-rate factor.

*Why it's done this way:* the gradient's direction carries local loss geometry, while its norm controls how much the optimizer will move. Clipping focuses on the norm because unstable training is often not about a wrong direction; it is about a direction whose length makes the update too large to trust.

### 2. Global-norm clipping rescales only unsafe gradients

The clipped gradient is

$$g_{clip}=g\cdot\min\left(1,\frac{c}{\lVert g\rVert}\right).$$

The multiplier is exactly 1 when the gradient norm is already at or below the threshold $c$. When the norm is larger than $c$, multiplying by $c/\lVert g\rVert$ shrinks the vector so its new length is exactly $c$.

In [ ]:
g_big_w = np.array([6.0, 8.0])  # length 10, same 3-4 direction scaled up.
c_w = 5.0  # maximum allowed gradient norm.
norm_big_w = np.linalg.norm(g_big_w)
scale_w = min(1.0, c_w / norm_big_w)
g_clip_w = g_big_w * scale_w
print("raw norm:", norm_big_w, "scale:", scale_w)
print("clipped gradient:", g_clip_w, "clipped norm:", np.linalg.norm(g_clip_w))
assert round(norm_big_w, 3) == 10.0
assert round(scale_w, 3) == 0.5
assert round(np.linalg.norm(g_clip_w), 3) == 5.0

▶ What you'll see: `[6, 8]` becomes `[3, 4]`, so the length is capped at 5 without turning the vector.

In [ ]:
g_small_w = np.array([1.0, 2.0])
scale_small_w = min(1.0, c_w / np.linalg.norm(g_small_w))
g_small_clip_w = g_small_w * scale_small_w
print("small norm:", round(np.linalg.norm(g_small_w), 3), "scale:", scale_small_w)
print("unchanged?", np.allclose(g_small_w, g_small_clip_w))
assert scale_small_w == 1.0

▶ What you'll see: a safe gradient is unchanged, because clipping is a cap rather than a universal shrinkage rule.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.quiver([0], [0], [g_big_w[0]], [g_big_w[1]], angles="xy", scale_units="xy", scale=1, color="crimson", label="raw g")
plt.quiver([0], [0], [g_clip_w[0]], [g_clip_w[1]], angles="xy", scale_units="xy", scale=1, color="seagreen", label="clipped g")
plt.xlim(0, 7); plt.ylim(0, 9)
plt.xlabel("parameter 0"); plt.ylabel("parameter 1")
plt.title("2: clipping preserves direction"); plt.legend(); plt.show()

▶ What you'll see: the green arrow lies on the same ray as the red arrow but stops at the threshold length.

*Why it's done this way:* using a single global multiplier preserves the vector's direction because every coordinate is scaled by the same number. That matters: the direction is the optimizer's best local estimate of how to reduce loss, while the threshold enforces a trust region on how far that estimate may move the parameters.

### 3. Clipping changes update size, not the learning-rate rule

Gradient descent still computes $\theta_{new}=\theta-\eta g$. Clipping simply replaces $g$ with $g_{clip}$ before the update. That means the learning rate remains the same knob, but the largest possible update length becomes $\eta c$.

In [ ]:
theta_w = np.array([2.0, -1.0])  # two parameters before the step.
eta_update_w = 0.2
raw_step_w = -eta_update_w * g_big_w
clip_step_w = -eta_update_w * g_clip_w
print("raw step length:", round(np.linalg.norm(raw_step_w), 3))
print("clipped step length:", round(np.linalg.norm(clip_step_w), 3))
assert round(np.linalg.norm(raw_step_w), 3) == 2.0
assert round(np.linalg.norm(clip_step_w), 3) == 1.0

▶ What you'll see: clipping halves the update length here, from 2.0 to the safe maximum `ηc = 1.0`.

In [ ]:
theta_raw_w = theta_w + raw_step_w
theta_clip_w = theta_w + clip_step_w
print("theta with raw gradient:", theta_raw_w)
print("theta with clipped gradient:", theta_clip_w)
print("distance between outcomes:", round(np.linalg.norm(theta_raw_w - theta_clip_w), 3))
assert round(np.linalg.norm(theta_raw_w - theta_clip_w), 3) == 1.0

▶ What you'll see: both updates move in the same direction, but the clipped endpoint is closer to the starting point.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.scatter(theta_w[0], theta_w[1], color="black", label="start θ")
plt.scatter(theta_raw_w[0], theta_raw_w[1], color="crimson", label="raw update")
plt.scatter(theta_clip_w[0], theta_clip_w[1], color="seagreen", label="clipped update")
plt.plot([theta_w[0], theta_raw_w[0]], [theta_w[1], theta_raw_w[1]], color="crimson", alpha=0.6)
plt.plot([theta_w[0], theta_clip_w[0]], [theta_w[1], theta_clip_w[1]], color="seagreen", alpha=0.8)
plt.title("3: same direction, safer endpoint"); plt.xlabel("θ0"); plt.ylabel("θ1"); plt.legend(); plt.show()

▶ What you'll see: the raw and clipped endpoints lie along the same line, but clipping limits how far the step can travel.

*Why it's done this way:* clipping before the update gives a hard bound on movement length without redesigning the optimizer. Since $\lVert -\eta g_{clip}\rVert\leq\eta c$, the threshold translates directly into a maximum parameter displacement per step.

### 4. Value clipping and norm clipping are different constraints

Sometimes people clip each coordinate into an interval, such as `[-c, c]`. That is value clipping. It is simple, but it can rotate the gradient because different coordinates may be cut by different amounts. Global-norm clipping instead rescales all coordinates together.

In [ ]:
g_coord_w = np.array([10.0, 1.0])
c_coord_w = 5.0
value_clip_w = np.clip(g_coord_w, -c_coord_w, c_coord_w)
norm_clip_w = g_coord_w * min(1.0, c_coord_w / np.linalg.norm(g_coord_w))
print("raw:", g_coord_w)
print("value clipped:", np.round(value_clip_w, 3), "norm", round(np.linalg.norm(value_clip_w), 3))
print("norm clipped:", np.round(norm_clip_w, 3), "norm", round(np.linalg.norm(norm_clip_w), 3))
assert np.allclose(value_clip_w, [5.0, 1.0])
assert round(np.linalg.norm(norm_clip_w), 3) == 5.0

▶ What you'll see: value clipping produces `[5, 1]`, while norm clipping produces a length-5 vector in the original direction.

In [ ]:
cos_value_w = float(np.dot(g_coord_w, value_clip_w) / (np.linalg.norm(g_coord_w) * np.linalg.norm(value_clip_w)))
cos_norm_w = float(np.dot(g_coord_w, norm_clip_w) / (np.linalg.norm(g_coord_w) * np.linalg.norm(norm_clip_w)))
print("cos(raw, value clipped):", round(cos_value_w, 4))
print("cos(raw, norm clipped):", round(cos_norm_w, 4))
assert round(cos_norm_w, 4) == 1.0

▶ What you'll see: norm clipping has cosine 1.0 with the raw gradient, while value clipping slightly changes the direction.

In [ ]:
plt.figure(figsize=(4, 3.5))
for vec_w, name_w, color_w in [(g_coord_w, "raw", "gray"), (value_clip_w, "value", "crimson"), (norm_clip_w, "norm", "seagreen")]:
    plt.quiver([0], [0], [vec_w[0]], [vec_w[1]], angles="xy", scale_units="xy", scale=1, label=name_w, color=color_w)
plt.xlim(0, 11); plt.ylim(0, 2)
plt.title("4: coordinate cap vs norm cap"); plt.xlabel("g0"); plt.ylabel("g1"); plt.legend(); plt.show()

▶ What you'll see: the norm-clipped arrow stays on the raw arrow's ray; the value-clipped arrow tilts upward relative to it.

*Why it's done this way:* a global norm constraint says, "trust this direction, but not this length." Coordinate clipping says, "no coordinate may be too large," which can be useful for numeric guards but changes the geometry of the descent direction.

### 5. Exploding gradients create unstable training steps

A tiny recurrent-style scalar example shows why spikes appear. If the same multiplier is applied repeatedly, the backward sensitivity multiplies repeatedly too. A derivative like $1.5^{10}$ is already much larger than one; deeper chains can become enormous.

In [ ]:
mult_w = 1.5
depths_w = np.arange(1, 13)
grads_chain_w = mult_w ** depths_w
print("first five chain sensitivities:", np.round(grads_chain_w[:5], 3))
print("depth 10 sensitivity:", round(float(grads_chain_w[9]), 3))
assert round(float(grads_chain_w[9]), 3) == 57.665

▶ What you'll see: repeated multiplication grows the gradient even in this one-number model.

In [ ]:
c_chain_w = 10.0
clipped_chain_w = np.minimum(grads_chain_w, c_chain_w)
print("raw max:", round(float(grads_chain_w.max()), 3), "clipped max:", clipped_chain_w.max())
assert clipped_chain_w.max() == 10.0

▶ What you'll see: the raw chain sensitivity keeps growing, but the clipped sequence never exceeds 10.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(depths_w, grads_chain_w, marker="o", label="raw chain gradient")
plt.plot(depths_w, clipped_chain_w, marker="s", label="clipped at 10")
plt.yscale("log")
plt.xlabel("chain depth"); plt.ylabel("gradient magnitude (log scale)")
plt.title("5: repeated derivatives can explode"); plt.legend(); plt.show()

▶ What you'll see: the raw curve rises exponentially on the log-scale plot, while clipping creates a flat ceiling.

*Why it's done this way:* deep models compose many derivatives. Products above one amplify backward signals, so a local gradient may reflect chain length as much as useful information. Clipping does not solve the underlying architecture, but it prevents one amplified signal from making an unsafe optimizer step.

### 6. Thresholds are training knobs, not magic constants

The threshold $c$ decides which gradients are left alone and which are capped. Too low clips ordinary learning signals and slows progress; too high clips only disasters. A practical threshold is chosen by inspecting gradient norms and deciding what counts as an outlier for the model and learning rate.

In [ ]:
rng_w = np.random.default_rng(0)
normal_norms_w = rng_w.lognormal(mean=0.0, sigma=0.35, size=120)
spikes_w = np.array([8.0, 12.0, 20.0])
norms_w = np.concatenate([normal_norms_w, spikes_w])
threshold_w = 3.0
clip_flags_w = norms_w > threshold_w
print("median norm:", round(float(np.median(norms_w)), 3))
print("clipped count:", int(clip_flags_w.sum()), "of", len(norms_w))
assert int(clip_flags_w.sum()) == 3

▶ What you'll see: the threshold clips the three injected spikes and leaves the ordinary lognormal norms alone.

In [ ]:
effective_norms_w = np.minimum(norms_w, threshold_w)
print("largest raw norm:", float(norms_w.max()))
print("largest effective norm:", float(effective_norms_w.max()))
assert effective_norms_w.max() == threshold_w

▶ What you'll see: the largest effective norm is exactly the threshold, even though the raw maximum is 20.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(norms_w, color="gray", label="raw norm")
plt.plot(effective_norms_w, color="seagreen", label="after clipping")
plt.axhline(threshold_w, color="crimson", linestyle="--", label="threshold")
plt.xlabel("training step"); plt.ylabel("gradient norm")
plt.title("6: clipping acts only on norm outliers"); plt.legend(); plt.show()

▶ What you'll see: most steps are untouched; only the spike peaks are flattened to the threshold line.

*Why it's done this way:* the threshold is a scale decision. Because an optimizer step has length at most $\eta c$, choosing $c$ should reflect both the model's typical gradient norms and the learning rate's safe movement budget.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for vectors, norms, dot products, and from-scratch optimization.
import matplotlib.pyplot as plt  # load Matplotlib for all plots in the examples.
np.random.seed(0)  # make every example deterministic.

def clip_by_global_norm(g, threshold):  # implement g * min(1, c / ||g||) from scratch.
    g = np.asarray(g, dtype=float)  # convert inputs to a numeric vector.
    norm = np.linalg.norm(g)  # measure the Euclidean gradient length.
    scale = 1.0 if norm == 0 else min(1.0, threshold / norm)  # avoid division by zero and cap only unsafe norms.
    return g * scale, norm, scale  # return the clipped vector plus diagnostic pieces.

def quadratic_loss(theta, target):  # simple convex loss used by several optimization demos.
    diff = theta - target  # residual from the optimum.
    return 0.5 * float(diff @ diff)  # half squared distance has gradient theta - target.

def quadratic_grad(theta, target):  # gradient of the quadratic loss above.
    return theta - target  # derivative of 0.5 ||theta-target||^2.

## 🟢 Basics (warm-up)

### Basic 1 — Measure a gradient norm

**Goal.** Compute the length of a gradient vector, because clipping starts by asking whether that length is safe. We build it in 2 steps.

In [ ]:
g_b1 = np.array([3.0, 4.0])  # define a two-coordinate gradient with known 3-4-5 length.
squares_b1 = g_b1 ** 2  # square coordinates because the L2 norm sums squared components.
print("squares:", squares_b1)  # inspect the pieces of the norm.

▶ What you'll see: the squared coordinates are 9 and 16.

In [ ]:
norm_b1 = np.linalg.norm(g_b1)  # compute sqrt(9 + 16).
print("gradient norm:", norm_b1)  # inspect the length before clipping.
assert norm_b1 == 5.0  # verify the concrete 3-4-5 triangle.
plt.figure(figsize=(4, 3))
plt.bar(["g0²", "g1²"], squares_b1, color="teal")
plt.title("Basic 1: squared pieces of ||g||")
plt.ylabel("squared value")
plt.show()

▶ What you'll see: the bars sum to 25, whose square root is the norm 5.

👀 Takeaway: gradient clipping is triggered by the vector length, not by a loss value directly.

### Basic 2 — Clip one large gradient

**Goal.** Apply the exact formula $g\min(1,c/\|g\|)$, because global-norm clipping caps length while preserving direction. We build it in 2 steps.

In [ ]:
g_b2 = np.array([6.0, 8.0])  # raw gradient with norm 10.
c_b2 = 5.0  # threshold used as the maximum allowed norm.
g_clip_b2, norm_b2, scale_b2 = clip_by_global_norm(g_b2, c_b2)  # clip by global norm.
print("raw norm:", norm_b2, "scale:", scale_b2)  # inspect the multiplier.
assert round(scale_b2, 3) == 0.5

▶ What you'll see: the scale is 0.5 because the raw norm is twice the threshold.

In [ ]:
print("clipped gradient:", g_clip_b2)  # inspect the rescaled gradient.
print("clipped norm:", np.linalg.norm(g_clip_b2))  # verify the norm cap.
assert np.allclose(g_clip_b2, [3.0, 4.0])
assert round(np.linalg.norm(g_clip_b2), 3) == 5.0
plt.figure(figsize=(4, 3))
plt.bar(["raw ||g||", "clipped ||g||"], [norm_b2, np.linalg.norm(g_clip_b2)], color=["crimson", "seagreen"])
plt.axhline(c_b2, color="black", linestyle="--")
plt.title("Basic 2: norm capped at c")
plt.show()

▶ What you'll see: the raw norm is above the dashed threshold; the clipped norm sits exactly on it.

👀 Takeaway: a too-large gradient is uniformly rescaled until its norm equals the threshold.

### Basic 3 — Leave a safe gradient unchanged

**Goal.** Show the `min(1, c / ||g||)` gate when the gradient is already safe, because clipping should not shrink ordinary learning signals. We build it in 2 steps.

In [ ]:
g_b3 = np.array([1.0, 2.0])  # raw gradient with norm below 5.
c_b3 = 5.0  # threshold larger than the current norm.
g_clip_b3, norm_b3, scale_b3 = clip_by_global_norm(g_b3, c_b3)  # apply the same clipping function.
print("raw norm:", round(norm_b3, 3), "scale:", scale_b3)  # inspect why nothing changes.
assert scale_b3 == 1.0

▶ What you'll see: the scale is exactly 1.0, so the vector passes through unchanged.

In [ ]:
print("same vector?", np.allclose(g_b3, g_clip_b3))  # confirm no modification happened.
assert np.allclose(g_b3, g_clip_b3)
plt.figure(figsize=(4, 3))
plt.bar(["g0", "g1"], g_clip_b3, color="steelblue")
plt.title("Basic 3: safe gradient is untouched")
plt.show()

▶ What you'll see: the clipped vector equals the original vector coordinate by coordinate.

👀 Takeaway: clipping is a safety cap, not a replacement for the learning rate.

### Basic 4 — Convert a clipped gradient into an update

**Goal.** Compare raw and clipped update lengths, because the optimizer moves by `-learning_rate × gradient`. We build it in 3 steps.

In [ ]:
g_b4 = np.array([6.0, 8.0])  # large gradient from Basic 2.
eta_b4 = 0.2  # learning rate for the parameter update.
g_clip_b4, norm_b4, scale_b4 = clip_by_global_norm(g_b4, 5.0)  # cap the gradient norm at 5.
print("raw norm:", norm_b4, "clipped norm:", np.linalg.norm(g_clip_b4))  # inspect before and after.

▶ What you'll see: the gradient length falls from 10 to 5 before the update is formed.

In [ ]:
raw_update_b4 = -eta_b4 * g_b4  # update using the unsafe raw gradient.
clip_update_b4 = -eta_b4 * g_clip_b4  # update using the clipped gradient.
print("raw update length:", round(np.linalg.norm(raw_update_b4), 3))
print("clipped update length:", round(np.linalg.norm(clip_update_b4), 3))
assert round(np.linalg.norm(clip_update_b4), 3) == 1.0

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["raw step", "clipped step"], [np.linalg.norm(raw_update_b4), np.linalg.norm(clip_update_b4)], color=["crimson", "seagreen"])
plt.title("Basic 4: clipping bounds update length")
plt.ylabel("step length")
plt.show()

▶ What you'll see: the clipped step length is `ηc = 0.2 × 5 = 1.0`.

👀 Takeaway: clipping creates a direct maximum step size of learning-rate times threshold.

### Basic 5 — Preserve the gradient direction

**Goal.** Verify that global-norm clipping does not rotate the gradient, because every coordinate receives the same multiplier. We build it in 2 steps.

In [ ]:
g_b5 = np.array([9.0, 12.0])  # another gradient in the 3-4 direction.
g_clip_b5, norm_b5, scale_b5 = clip_by_global_norm(g_b5, 5.0)  # clip length from 15 down to 5.
cos_b5 = float(np.dot(g_b5, g_clip_b5) / (np.linalg.norm(g_b5) * np.linalg.norm(g_clip_b5)))  # cosine of raw and clipped directions.
print("scale:", round(scale_b5, 3), "cosine:", round(cos_b5, 6))  # inspect direction preservation.
assert round(cos_b5, 6) == 1.0

▶ What you'll see: cosine equals 1, meaning the two vectors point in the same direction.

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.quiver([0], [0], [g_b5[0]], [g_b5[1]], angles="xy", scale_units="xy", scale=1, color="gray", label="raw")
plt.quiver([0], [0], [g_clip_b5[0]], [g_clip_b5[1]], angles="xy", scale_units="xy", scale=1, color="seagreen", label="clipped")
plt.xlim(0, 10); plt.ylim(0, 13)
plt.title("Basic 5: same ray after clipping")
plt.legend(); plt.show()

▶ What you'll see: the clipped arrow lies exactly on top of the raw arrow's direction, just shorter.

👀 Takeaway: global-norm clipping constrains magnitude while retaining the optimizer's local direction estimate.

### Basic 6 — Clip a zero gradient safely

**Goal.** Handle the zero vector without division by zero, because a flat local gradient should stay zero. We build it in 2 steps.

In [ ]:
g_b6 = np.array([0.0, 0.0, 0.0])  # zero gradient has no direction and no length.
g_clip_b6, norm_b6, scale_b6 = clip_by_global_norm(g_b6, 1.0)  # apply guarded clipping.
print("norm:", norm_b6, "scale:", scale_b6)  # inspect the guard path.
assert norm_b6 == 0.0

▶ What you'll see: the norm is zero and the helper returns scale 1 rather than dividing by zero.

In [ ]:
print("clipped zero:", g_clip_b6)  # inspect the unchanged zero vector.
assert np.allclose(g_clip_b6, np.zeros(3))
plt.figure(figsize=(4, 3))
plt.bar(["g0", "g1", "g2"], g_clip_b6, color="gray")
plt.ylim(-1, 1)
plt.title("Basic 6: zero remains zero")
plt.show()

▶ What you'll see: all bars stay at zero.

👀 Takeaway: robust clipping code must explicitly handle zero norm.

### Basic 7 — Compare norm clipping to value clipping

**Goal.** See why coordinate-wise clipping is not the same as global-norm clipping. We build it in 3 steps.

In [ ]:
g_b7 = np.array([10.0, 1.0])  # one coordinate is much larger than the other.
c_b7 = 5.0  # clipping threshold.
value_b7 = np.clip(g_b7, -c_b7, c_b7)  # coordinate-wise cap.
norm_b7, raw_norm_b7, scale_b7 = clip_by_global_norm(g_b7, c_b7)  # global-norm cap.
print("value clipped:", value_b7)
print("norm clipped:", np.round(norm_b7, 3))

▶ What you'll see: the two methods produce different vectors.

In [ ]:
cos_value_b7 = float(np.dot(g_b7, value_b7) / (np.linalg.norm(g_b7) * np.linalg.norm(value_b7)))
cos_norm_b7 = float(np.dot(g_b7, norm_b7) / (np.linalg.norm(g_b7) * np.linalg.norm(norm_b7)))
print("cosines:", round(cos_value_b7, 4), round(cos_norm_b7, 4))
assert round(cos_norm_b7, 4) == 1.0

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["raw", "value", "norm"], [np.linalg.norm(g_b7), np.linalg.norm(value_b7), np.linalg.norm(norm_b7)], color=["gray", "crimson", "seagreen"])
plt.axhline(c_b7, linestyle="--", color="black")
plt.title("Basic 7: two clipping rules")
plt.ylabel("vector norm")
plt.show()

▶ What you'll see: value clipping still leaves a norm above 5, while norm clipping lands exactly at 5.

👀 Takeaway: value clipping caps coordinates; norm clipping caps the whole update direction.

### Basic 8 — Clip a list of gradient norms

**Goal.** Apply clipping to many steps, because training stability is about repeated updates rather than one vector. We build it in 2 steps.

In [ ]:
norms_b8 = np.array([0.8, 1.1, 9.0, 1.4, 12.0, 0.9])  # gradient norms across six training steps.
c_b8 = 3.0  # threshold for this toy run.
effective_b8 = np.minimum(norms_b8, c_b8)  # the norm after clipping.
print("effective norms:", effective_b8)
assert np.all(effective_b8 <= c_b8)

▶ What you'll see: only the 9 and 12 spikes are capped to 3.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(norms_b8, marker="o", label="raw")
plt.plot(effective_b8, marker="s", label="after clipping")
plt.axhline(c_b8, color="crimson", linestyle="--", label="threshold")
plt.title("Basic 8: norm spikes are flattened")
plt.xlabel("step"); plt.ylabel("norm"); plt.legend(); plt.show()

▶ What you'll see: ordinary steps are untouched and the two spikes flatten at the threshold.

👀 Takeaway: clipping is usually invisible until a norm spike appears.

### Basic 9 — Clip a scalar gradient

**Goal.** Connect vector clipping to a one-parameter model, because in one dimension norm clipping becomes absolute-value clipping. We build it in 2 steps.

In [ ]:
g_b9 = np.array([7.0])  # scalar gradient represented as a length-one vector.
c_b9 = 2.0  # maximum absolute gradient.
g_clip_b9, norm_b9, scale_b9 = clip_by_global_norm(g_b9, c_b9)  # apply the same vector rule.
print("raw:", g_b9[0], "clipped:", g_clip_b9[0], "scale:", round(scale_b9, 3))
assert float(g_clip_b9[0]) == 2.0

▶ What you'll see: the scalar 7 is clipped down to 2.

In [ ]:
theta_b9 = 1.0
eta_b9 = 0.5
theta_new_b9 = theta_b9 - eta_b9 * g_clip_b9[0]
print("new theta:", theta_new_b9)
assert theta_new_b9 == 0.0
plt.figure(figsize=(4, 3))
plt.bar(["raw scalar g", "clipped scalar g"], [g_b9[0], g_clip_b9[0]], color=["crimson", "seagreen"])
plt.title("Basic 9: scalar clipping")
plt.show()

▶ What you'll see: the clipped scalar produces a controlled update from 1 to 0.

👀 Takeaway: global-norm clipping naturally reduces to absolute-value clipping for a single parameter.

### Basic 10 — Inspect a mini training step

**Goal.** Use clipping inside one quadratic-gradient update, because this is the smallest full optimizer example. We build it in 3 steps.

In [ ]:
theta_b10 = np.array([10.0, -6.0])  # current parameters far from the target.
target_b10 = np.array([1.0, 2.0])  # optimum of the quadratic loss.
g_b10 = quadratic_grad(theta_b10, target_b10)  # gradient equals theta - target.
print("loss before:", quadratic_loss(theta_b10, target_b10))
print("raw gradient:", g_b10, "norm:", round(np.linalg.norm(g_b10), 3))

▶ What you'll see: the far-away parameters create a large gradient.

In [ ]:
g_clip_b10, norm_b10, scale_b10 = clip_by_global_norm(g_b10, 5.0)  # cap the gradient length.
theta_next_b10 = theta_b10 - 0.2 * g_clip_b10  # take one clipped gradient descent step.
print("scale:", round(scale_b10, 3), "theta next:", np.round(theta_next_b10, 3))

In [ ]:
loss_after_b10 = quadratic_loss(theta_next_b10, target_b10)  # evaluate whether the step helped.
print("loss after:", round(loss_after_b10, 3))
assert loss_after_b10 < quadratic_loss(theta_b10, target_b10)
plt.figure(figsize=(4, 3))
plt.bar(["before", "after"], [quadratic_loss(theta_b10, target_b10), loss_after_b10], color=["gray", "seagreen"])
plt.title("Basic 10: clipped step lowers loss")
plt.ylabel("quadratic loss")
plt.show()

▶ What you'll see: even after capping the step, the loss decreases.

👀 Takeaway: clipping can preserve useful descent while preventing an oversized update.

## 🟡 Easy

### Easy 1 — Implement the clipping formula as diagnostics

**Goal.** Return the clipped gradient, raw norm, scale, and clipped norm, because training logs should show when clipping actually activates. We build it in 3 steps.

In [ ]:
g_e1 = np.array([2.0, -1.0, 10.0])  # a three-parameter gradient with one large coordinate.
c_e1 = 4.0  # global norm threshold.
g_clip_e1, norm_e1, scale_e1 = clip_by_global_norm(g_e1, c_e1)  # compute all clipping diagnostics.
print("raw norm:", round(norm_e1, 3), "scale:", round(scale_e1, 3))
assert norm_e1 > c_e1

▶ What you'll see: the raw norm exceeds the threshold, so the scale is below 1.

In [ ]:
clip_norm_e1 = np.linalg.norm(g_clip_e1)  # compute the post-clipping norm.
print("clipped vector:", np.round(g_clip_e1, 3))
print("clipped norm:", round(clip_norm_e1, 3))
assert round(clip_norm_e1, 3) == 4.0

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["raw", "clipped"], [norm_e1, clip_norm_e1], color=["crimson", "seagreen"])
plt.axhline(c_e1, color="black", linestyle="--")
plt.title("Easy 1: clipping diagnostics")
plt.ylabel("gradient norm")
plt.show()

▶ What you'll see: the clipped bar sits on the threshold, and the raw bar rises above it.

👀 Takeaway: logging norm and scale tells you whether clipping is rare safety behavior or constant shrinkage.

### Easy 2 — Train a quadratic with and without clipping

**Goal.** Compare optimization paths when the initial gradient is large, because clipping should make early steps less violent. We build it in 4 steps.

In [ ]:
target_e2 = np.array([0.0, 0.0])  # optimum at the origin.
theta_raw_e2 = np.array([8.0, 6.0])  # raw-gradient run starts far away.
theta_clip_e2 = theta_raw_e2.copy()  # clipped run starts from the same point.
eta_e2 = 0.35  # deliberately large learning rate for a visible contrast.
print("initial loss:", quadratic_loss(theta_raw_e2, target_e2))

▶ What you'll see: both runs begin with the same high loss.

In [ ]:
losses_raw_e2 = []
losses_clip_e2 = []
for step_e2 in range(12):
    g_raw_e2 = quadratic_grad(theta_raw_e2, target_e2)  # raw gradient for the unclipped run.
    theta_raw_e2 = theta_raw_e2 - eta_e2 * g_raw_e2  # take the raw step.
    g_now_e2 = quadratic_grad(theta_clip_e2, target_e2)  # raw gradient for the clipped run.
    g_safe_e2, _, _ = clip_by_global_norm(g_now_e2, 3.0)  # cap its norm.
    theta_clip_e2 = theta_clip_e2 - eta_e2 * g_safe_e2  # take the clipped step.
    losses_raw_e2.append(quadratic_loss(theta_raw_e2, target_e2))
    losses_clip_e2.append(quadratic_loss(theta_clip_e2, target_e2))
print("first raw/clipped loss:", round(losses_raw_e2[0], 3), round(losses_clip_e2[0], 3))

In [ ]:
print("final raw loss:", round(losses_raw_e2[-1], 5))
print("final clipped loss:", round(losses_clip_e2[-1], 5))
assert losses_raw_e2[-1] < losses_raw_e2[0]
assert losses_clip_e2[-1] < losses_clip_e2[0]

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_raw_e2, marker="o", label="raw gradients")
plt.plot(losses_clip_e2, marker="s", label="clipped gradients")
plt.yscale("log")
plt.title("Easy 2: two quadratic training paths")
plt.xlabel("step"); plt.ylabel("loss (log scale)"); plt.legend(); plt.show()

▶ What you'll see: both curves descend, but the clipped run limits the largest early movement.

👀 Takeaway: clipping is most visible when early gradients are much larger than the desired step budget.

### Easy 3 — Clip a minibatch gradient average

**Goal.** Average per-example gradients and then clip the batch gradient, because minibatch optimizers usually update from an aggregate gradient. We build it in 3 steps.

In [ ]:
grads_e3 = np.array([[1.0, 0.5], [0.8, 0.4], [20.0, 10.0], [1.2, 0.6]])  # one example is an outlier.
batch_grad_e3 = grads_e3.mean(axis=0)  # ordinary minibatch gradient.
print("batch gradient:", batch_grad_e3)
print("batch norm:", round(np.linalg.norm(batch_grad_e3), 3))

▶ What you'll see: the outlier pulls the average gradient upward.

In [ ]:
g_clip_e3, norm_e3, scale_e3 = clip_by_global_norm(batch_grad_e3, 3.0)  # clip the aggregate gradient.
print("scale:", round(scale_e3, 3), "clipped batch gradient:", np.round(g_clip_e3, 3))
assert round(np.linalg.norm(g_clip_e3), 3) == 3.0

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(grads_e3[:, 0], grads_e3[:, 1], color="gray", label="per-example")
plt.scatter([batch_grad_e3[0]], [batch_grad_e3[1]], color="crimson", label="batch mean")
plt.scatter([g_clip_e3[0]], [g_clip_e3[1]], color="seagreen", label="clipped mean")
plt.title("Easy 3: outlier affects the batch gradient")
plt.xlabel("g0"); plt.ylabel("g1"); plt.legend(); plt.show()

▶ What you'll see: the clipped batch mean lies between the origin and the outlier-influenced mean.

👀 Takeaway: clipping after averaging caps the actual update the optimizer will apply.

### Easy 4 — Sweep thresholds for the same gradient

**Goal.** See how threshold choice controls the effective gradient norm, because a too-small threshold clips almost everything. We build it in 3 steps.

In [ ]:
g_e4 = np.array([6.0, 8.0])  # norm 10 gradient.
thresholds_e4 = np.array([1.0, 2.0, 5.0, 10.0, 20.0])  # candidate clipping thresholds.
effective_e4 = []
for c_e4 in thresholds_e4:
    clipped_e4, _, _ = clip_by_global_norm(g_e4, c_e4)
    effective_e4.append(np.linalg.norm(clipped_e4))
effective_e4 = np.array(effective_e4)
print("effective norms:", effective_e4)

▶ What you'll see: effective norm equals the threshold until the threshold reaches the raw norm.

In [ ]:
scales_e4 = effective_e4 / np.linalg.norm(g_e4)  # scale applied relative to the raw gradient.
print("scales:", np.round(scales_e4, 3))
assert np.allclose(effective_e4, [1, 2, 5, 10, 10])

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(thresholds_e4, effective_e4, marker="o", color="purple")
plt.axhline(np.linalg.norm(g_e4), color="gray", linestyle="--", label="raw norm")
plt.title("Easy 4: threshold vs effective norm")
plt.xlabel("threshold c"); plt.ylabel("||g_clip||"); plt.legend(); plt.show()

▶ What you'll see: the curve rises with the threshold and then saturates at the raw norm 10.

👀 Takeaway: the threshold is a direct cap on gradient norm, so it should be tuned to the scale of the run.

### Easy 5 — Monitor clipping frequency

**Goal.** Count how often clipping activates, because constant clipping may mean the threshold is too low or the learning dynamics are unstable. We build it in 3 steps.

In [ ]:
norms_e5 = np.array([0.9, 1.2, 1.1, 6.0, 0.8, 7.5, 1.0, 1.3, 9.0, 0.7])  # logged gradient norms.
c_e5 = 3.0  # candidate clipping threshold.
active_e5 = norms_e5 > c_e5  # clipping activates only above threshold.
print("clipped steps:", np.where(active_e5)[0])
assert int(active_e5.sum()) == 3

▶ What you'll see: three step indices are above the threshold.

In [ ]:
frequency_e5 = active_e5.mean()  # fraction of steps that were clipped.
print("clipping frequency:", round(float(frequency_e5), 3))
assert round(float(frequency_e5), 3) == 0.3

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(norms_e5)), norms_e5, color=np.where(active_e5, "crimson", "steelblue"))
plt.axhline(c_e5, color="black", linestyle="--")
plt.title("Easy 5: clipping frequency")
plt.xlabel("step"); plt.ylabel("gradient norm"); plt.show()

▶ What you'll see: red bars mark the 30% of steps that would be clipped.

👀 Takeaway: clipping frequency is a practical diagnostic for threshold quality.

## 🔴 Advanced

### Advanced 1 — Clipped gradient descent on a steep valley

**Goal.** Stabilize a steep two-dimensional quadratic, because different curvature along different axes can create large gradients. We build it in 4 steps.

In [ ]:
def steep_loss_a1(theta_a1):
    return 0.5 * (25.0 * theta_a1[0] ** 2 + theta_a1[1] ** 2)

def steep_grad_a1(theta_a1):
    return np.array([25.0 * theta_a1[0], theta_a1[1]])

theta_raw_a1 = np.array([2.0, 2.0])
theta_clip_a1 = theta_raw_a1.copy()
eta_a1 = 0.08
print("initial gradient:", steep_grad_a1(theta_raw_a1))

▶ What you'll see: the first coordinate's gradient is much larger because that axis is steep.

In [ ]:
raw_path_a1 = [theta_raw_a1.copy()]
clip_path_a1 = [theta_clip_a1.copy()]
for step_a1 in range(25):
    theta_raw_a1 = theta_raw_a1 - eta_a1 * steep_grad_a1(theta_raw_a1)
    g_a1 = steep_grad_a1(theta_clip_a1)
    g_safe_a1, _, _ = clip_by_global_norm(g_a1, 6.0)
    theta_clip_a1 = theta_clip_a1 - eta_a1 * g_safe_a1
    raw_path_a1.append(theta_raw_a1.copy())
    clip_path_a1.append(theta_clip_a1.copy())
raw_path_a1 = np.array(raw_path_a1)
clip_path_a1 = np.array(clip_path_a1)
print("final clipped loss:", round(steep_loss_a1(theta_clip_a1), 4))

In [ ]:
loss_raw_a1 = np.array([steep_loss_a1(x_a1) for x_a1 in raw_path_a1])
loss_clip_a1 = np.array([steep_loss_a1(x_a1) for x_a1 in clip_path_a1])
print("raw finite?", np.all(np.isfinite(loss_raw_a1)), "clipped final lower than start?", loss_clip_a1[-1] < loss_clip_a1[0])
assert loss_clip_a1[-1] < loss_clip_a1[0]

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.plot(raw_path_a1[:, 0], raw_path_a1[:, 1], marker="o", label="raw")
plt.plot(clip_path_a1[:, 0], clip_path_a1[:, 1], marker="s", label="clipped")
plt.scatter([0], [0], color="black", label="minimum")
plt.title("Advanced 1: paths in a steep valley")
plt.xlabel("θ0"); plt.ylabel("θ1"); plt.legend(); plt.show()

▶ What you'll see: clipping limits the most extreme motion along the steep axis.

👀 Takeaway: norm clipping is a trust-region style guard for high-curvature or high-scale gradients.

### Advanced 2 — Per-example clipping before averaging

**Goal.** Compare clipping each example before averaging with clipping only the final batch average, because differential privacy and robust training often need per-example caps. We build it in 4 steps.

In [ ]:
grads_a2 = np.array([[1.0, 0.0], [1.0, 0.2], [30.0, 0.0], [1.0, -0.2]])  # one extreme per-example gradient.
c_a2 = 2.0  # per-example or batch threshold.
batch_mean_a2 = grads_a2.mean(axis=0)  # average first.
print("raw batch mean:", batch_mean_a2)

▶ What you'll see: the single outlier dominates the first coordinate of the mean.

In [ ]:
batch_clipped_a2, _, _ = clip_by_global_norm(batch_mean_a2, c_a2)  # clip after averaging.
per_example_clipped_a2 = np.array([clip_by_global_norm(row_a2, c_a2)[0] for row_a2 in grads_a2])  # clip each row first.
mean_per_clipped_a2 = per_example_clipped_a2.mean(axis=0)  # then average.
print("clip after mean:", np.round(batch_clipped_a2, 3))
print("mean after per-example clip:", np.round(mean_per_clipped_a2, 3))

In [ ]:
diff_a2 = np.linalg.norm(batch_clipped_a2 - mean_per_clipped_a2)
print("difference between strategies:", round(diff_a2, 3))
assert diff_a2 > 0.5

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["avg then clip", "clip then avg"], [np.linalg.norm(batch_clipped_a2), np.linalg.norm(mean_per_clipped_a2)], color=["crimson", "seagreen"])
plt.title("Advanced 2: two minibatch clipping orders")
plt.ylabel("effective update norm")
plt.show()

▶ What you'll see: per-example clipping reduces the outlier's influence before it can dominate the batch mean.

👀 Takeaway: clipping order changes the estimator, so choose batch clipping or per-example clipping according to the training goal.

### Advanced 3 — Clipping with momentum

**Goal.** Study whether to clip the current gradient before feeding momentum, because momentum can accumulate old spikes even after the current gradient is safe. We build it in 4 steps.

In [ ]:
grads_a3 = np.array([[1.0, 0.0], [1.0, 0.0], [20.0, 0.0], [1.0, 0.0], [1.0, 0.0]])  # one spike in a sequence.
beta_a3 = 0.9  # momentum coefficient.
c_a3 = 3.0  # gradient clipping threshold.
v_raw_a3 = np.zeros(2)
v_clip_a3 = np.zeros(2)
print("sequence length:", len(grads_a3))

▶ What you'll see: the toy sequence has one large spike in the middle.

In [ ]:
vel_raw_a3 = []
vel_clip_a3 = []
for g_a3 in grads_a3:
    v_raw_a3 = beta_a3 * v_raw_a3 + g_a3  # momentum receives the raw gradient.
    g_safe_a3, _, _ = clip_by_global_norm(g_a3, c_a3)  # current gradient is capped before momentum.
    v_clip_a3 = beta_a3 * v_clip_a3 + g_safe_a3  # momentum receives the clipped gradient.
    vel_raw_a3.append(np.linalg.norm(v_raw_a3))
    vel_clip_a3.append(np.linalg.norm(v_clip_a3))
print("velocity norms raw:", np.round(vel_raw_a3, 3))
print("velocity norms clipped-gradient:", np.round(vel_clip_a3, 3))

In [ ]:
assert vel_raw_a3[2] > vel_clip_a3[2]
print("spike velocity ratio:", round(vel_raw_a3[2] / vel_clip_a3[2], 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(vel_raw_a3, marker="o", label="momentum on raw g")
plt.plot(vel_clip_a3, marker="s", label="momentum on clipped g")
plt.title("Advanced 3: clipping before momentum")
plt.xlabel("step"); plt.ylabel("velocity norm"); plt.legend(); plt.show()

▶ What you'll see: the raw-momentum velocity jumps much higher when the spike arrives and decays slowly afterward.

👀 Takeaway: clipping the gradient before momentum prevents one spike from being stored in the optimizer's memory.

### Advanced 4 — Threshold from a percentile of observed norms

**Goal.** Choose a data-driven threshold from historical norms, because the right cap depends on model scale and batch size. We build it in 3 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)
norms_a4 = rng_a4.lognormal(mean=0.2, sigma=0.45, size=200)
norms_a4 = np.concatenate([norms_a4, np.array([9.0, 11.0, 14.0])])
threshold_a4 = float(np.percentile(norms_a4, 95))
print("95th percentile threshold:", round(threshold_a4, 3))
assert threshold_a4 > np.median(norms_a4)

▶ What you'll see: the threshold is above the median but below the largest spikes.

In [ ]:
active_a4 = norms_a4 > threshold_a4
print("clipped fraction:", round(float(active_a4.mean()), 3))
assert 0.04 <= active_a4.mean() <= 0.06

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(norms_a4, bins=30, color="steelblue", alpha=0.8)
plt.axvline(threshold_a4, color="crimson", linestyle="--", label="95th percentile")
plt.title("Advanced 4: data-driven clipping threshold")
plt.xlabel("gradient norm"); plt.ylabel("count"); plt.legend(); plt.show()

▶ What you'll see: the vertical line sits near the right tail, clipping only unusually large norms.

👀 Takeaway: percentile thresholds make clipping a tail-risk control rather than a constant shrink on every step.

### Advanced 5 — Prevent a spike from ruining a training curve

**Goal.** Simulate noisy gradients with one large spike, because clipping is most useful when rare unstable steps would otherwise dominate the trajectory. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(5)
true_grad_a5 = np.array([1.0, -0.5])  # underlying descent direction.
noise_a5 = 0.2 * rng_a5.normal(size=(40, 2))  # ordinary stochastic gradient noise.
grads_a5 = true_grad_a5 + noise_a5
grads_a5[18] = np.array([-30.0, 15.0])  # inject one exploding gradient spike pointing the wrong way.
print("spike norm:", round(np.linalg.norm(grads_a5[18]), 3))
assert np.linalg.norm(grads_a5[18]) > 30

▶ What you'll see: one gradient norm is far larger than the rest.

In [ ]:
theta_raw_a5 = np.array([5.0, -3.0])
theta_clip_a5 = theta_raw_a5.copy()
target_a5 = np.array([0.0, 0.0])
eta_a5 = 0.1
raw_losses_a5 = []
clip_losses_a5 = []
print("start loss:", quadratic_loss(theta_raw_a5, target_a5))

In [ ]:
for g_a5 in grads_a5:
    theta_raw_a5 = theta_raw_a5 - eta_a5 * g_a5  # raw stochastic step.
    g_safe_a5, _, _ = clip_by_global_norm(g_a5, 3.0)  # cap only the sampled gradient.
    theta_clip_a5 = theta_clip_a5 - eta_a5 * g_safe_a5  # clipped stochastic step.
    raw_losses_a5.append(quadratic_loss(theta_raw_a5, target_a5))
    clip_losses_a5.append(quadratic_loss(theta_clip_a5, target_a5))
print("loss at spike raw/clipped:", round(raw_losses_a5[18], 3), round(clip_losses_a5[18], 3))

In [ ]:
raw_jump_a5 = raw_losses_a5[18] - raw_losses_a5[17]
clip_jump_a5 = clip_losses_a5[18] - clip_losses_a5[17]
print("spike loss jump raw:", round(raw_jump_a5, 3), "clipped:", round(clip_jump_a5, 3))
assert raw_jump_a5 > clip_jump_a5

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(raw_losses_a5, label="raw noisy gradients", color="crimson")
plt.plot(clip_losses_a5, label="clipped noisy gradients", color="seagreen")
plt.axvline(18, color="black", linestyle="--", label="spike")
plt.title("Advanced 5: clipping protects against one spike")
plt.xlabel("step"); plt.ylabel("quadratic loss"); plt.legend(); plt.show()

▶ What you'll see: the raw curve jumps upward at the spike, while the clipped curve is protected by the norm cap.

👀 Takeaway: gradient clipping is a stability guard for rare but high-impact gradient spikes, especially in deep composed models.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Clipping caps the update direction's norm without changing its direction when gradients become unsafe.

When rare gradients become too large, one step can dominate training. Norm clipping rescales the gradient vector only when its length exceeds a threshold. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, standardize, predict, and return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def one_hot(y, k):
    out = np.zeros((len(y), k))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def random_relu_features(X, seed=0, width=24):
    rng = np.random.default_rng(seed + X.shape[1])
    W = rng.normal(0.0, 1.0 / np.sqrt(max(1, X.shape[1])), size=(X.shape[1], width))
    b = rng.normal(0.0, 0.15, size=width)
    H = np.maximum(0.0, X @ W + b)
    pair = X[:, :1] * X[:, 1:2] if X.shape[1] >= 2 else X
    return np.hstack([X, X * X, pair, H])


def batch_norm_fit(H, eps=1e-5):
    mu = H.mean(axis=0, keepdims=True)
    var = H.var(axis=0, keepdims=True)
    Z = (H - mu) / np.sqrt(var + eps)
    return Z, (mu, var, eps)


def batch_norm_apply(H, params):
    mu, var, eps = params
    return (H - mu) / np.sqrt(var + eps)


def layer_norm(H, eps=1e-5):
    mu = H.mean(axis=1, keepdims=True)
    var = H.var(axis=1, keepdims=True)
    return (H - mu) / np.sqrt(var + eps)


def group_norm(H, groups=4, eps=1e-5):
    usable = (H.shape[1] // groups) * groups
    head = H[:, :usable].reshape(H.shape[0], groups, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def instance_norm(H, eps=1e-5):
    usable = (H.shape[1] // 8) * 8
    head = H[:, :usable].reshape(H.shape[0], 8, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def deep_random_features(X, depth=4, scale=1.0, residual=False, seed=0):
    H = random_relu_features(X, seed=seed, width=20)
    rng = np.random.default_rng(seed + 100 + H.shape[1])
    for _ in range(depth):
        W = rng.normal(0.0, scale / np.sqrt(H.shape[1]), size=(H.shape[1], H.shape[1]))
        F = np.maximum(0.0, H @ W)
        if residual:
            H = H + 0.35 * F
        else:
            H = F
    return H


def transform_pair(x_tr, x_te, mode="plain", seed=0, scale=1.0, residual=False):
    Htr = random_relu_features(x_tr, seed=seed)
    Hte = random_relu_features(x_te, seed=seed)
    if mode == "batchnorm":
        Htr, params = batch_norm_fit(Htr)
        Hte = batch_norm_apply(Hte, params)
    if mode == "test_batchnorm_wrong":
        Htr, params = batch_norm_fit(Htr)
        Hte, _ = batch_norm_fit(Hte)
    if mode == "layernorm":
        Htr = layer_norm(Htr)
        Hte = layer_norm(Hte)
    if mode == "groupnorm":
        Htr = group_norm(Htr)
        Hte = group_norm(Hte)
    if mode == "instancenorm":
        Htr = instance_norm(Htr)
        Hte = instance_norm(Hte)
    if mode == "deep":
        Htr = deep_random_features(x_tr, depth=5, scale=scale, residual=residual, seed=seed)
        Hte = deep_random_features(x_te, depth=5, scale=scale, residual=residual, seed=seed)
    return Htr, Hte


def train_softmax_classifier(x_tr, y_tr, x_te, epsilon=0.0, epochs=40, lr=0.2, clip=None, schedule="constant", transform="plain", seed=0, scale=1.0, residual=False):
    Htr, Hte = transform_pair(x_tr, x_te, mode=transform, seed=seed, scale=scale, residual=residual)
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 700)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    targets = (1.0 - epsilon) * Y + epsilon / k
    rng = np.random.default_rng(seed + 10)
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    losses = []
    grad_norms = []
    for epoch in range(epochs):
        eta = lr_value(schedule, epoch, epochs, lr)
        P = softmax(Htr @ W + b)
        loss = -np.mean(np.sum(targets * np.log(P + 1e-12), axis=1))
        G = (P - targets) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        norm = float(np.sqrt(np.sum(dW * dW) + np.sum(db * db)))
        if clip is not None:
            factor = min(1.0, clip / (norm + 1e-12))
            dW = dW * factor
            db = db * factor
        W = W - eta * dW
        b = b - eta * db
        losses.append(float(loss))
        grad_norms.append(norm)
    preds = np.argmax(Hte @ W + b, axis=1)
    return preds, losses, grad_norms


def lr_value(schedule, epoch, epochs, base):
    if schedule == "constant":
        return base
    if schedule == "step":
        return base if epoch < epochs // 2 else base * 0.2
    if schedule == "cosine":
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * epoch / max(1, epochs - 1)))
    if schedule == "warmup_cosine":
        warm = max(2, epochs // 5)
        if epoch < warm:
            return base * (epoch + 1) / warm
        span = max(1, epochs - warm - 1)
        t = epoch - warm
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * t / span))
    if schedule == "onecycle":
        half = max(1, epochs // 2)
        if epoch < half:
            return base * (0.2 + 1.8 * epoch / half)
        return base * (2.0 - 1.8 * (epoch - half) / max(1, epochs - half))
    return base


def component_accuracy(name, X, y, **kwargs):
    def build(x_tr, y_tr, x_te):
        preds, _, _ = train_softmax_classifier(x_tr, y_tr, x_te, **kwargs)
        return preds
    return clf_accuracy(build, X, y)


def fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.3, seed=0):
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 701)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    rng = np.random.default_rng(seed + Htr.shape[1])
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    for epoch in range(epochs):
        P = softmax(Htr @ W + b)
        G = (P - Y) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        W = W - lr * dW
        b = b - lr * db
    return np.argmax(Hte @ W + b, axis=1)


def logistic_accuracy_for_features(X, y, mode="plain", scale=1.0, residual=False, seed=0):
    def build(x_tr, y_tr, x_te):
        Htr, Hte = transform_pair(x_tr, x_te, mode=mode, seed=seed, scale=scale, residual=residual)
        return fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.35, seed=seed)
    return clf_accuracy(build, X, y)


def ladder_preview(rungs):
    for name, X, y in rungs:
        classes = np.unique(y)
        print(f"{name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")
    print("First D1 sample:", rungs[0][1][0].tolist(), "label=", int(rungs[0][2][0]))


def print_metric_table(rows, header="rung metric"):
    print(header)
    for name, metric in rows:
        print(f"{name:36s} {metric:.3f}")


def split_for_demo(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def plot_ladder_results(rungs, metrics, title, artifact_fn=None):
    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, (name, X, y) in zip(axes, rungs):
        if artifact_fn is None:
            if X.shape[1] == 64:
                ax.imshow(X[0].reshape(8, 8), cmap="gray")
            else:
                ax.scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=12)
        else:
            artifact_fn(ax, name, X, y)
        ax.set_title(name.split()[0])
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title + " artifacts")
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(range(1, 6), metrics, marker="o")
    plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylim(0.0, 1.05)
    plt.ylabel("held-out accuracy")
    plt.title(title + " summary")
    plt.grid(True, alpha=0.3)
    plt.show()

## The concept, built once

The lesson formula is $g_{clip}=g\cdot\min\left(1,\frac{c}{\|g\|}\right)$. For $g=[3,4]$ and threshold $c=2$, the raw norm is $5$, the multiplier is $0.4$, and the clipped vector is $[1.2,1.6]$.

In [ ]:
g = np.array([3.0, 4.0])
c = 2.0
norm = np.linalg.norm(g)
factor = min(1.0, c / norm)
g_clip = g * factor
print("raw norm:", norm, "factor:", factor, "clipped:", g_clip)
assert np.isclose(norm, 5.0)
assert np.allclose(g_clip, np.array([1.2, 1.6]))

This helper is the reusable method for the rest of the notebook. It keeps the model and ladder fixed, then varies only this topic's component.

In [ ]:
print('Reusable component method is available in the setup cell and verified above.')

## The dataset ladder

We use the shared F5 classification ladder: D1 XOR, D2 blobs, D3 noisy moons, D4 real sklearn digits, and D5 noisy digits. The same accuracy wrapper and model family run on every rung.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1–D5

The table reports one held-out accuracy per rung while the component-specific sweep is printed for auditability.

In [ ]:
rungs = clf_digits_ladder()
rows = []
for rung_id, (name, X, y) in enumerate(rungs):
    unclipped = component_accuracy(name, X, y, epsilon=0.0, epochs=40, lr=0.6, clip=None, seed=70 + rung_id)
    clipped = component_accuracy(name, X, y, epsilon=0.0, epochs=40, lr=0.6, clip=1.0, seed=70 + rung_id)
    rows.append((name, clipped))
    print(name, "unclipped/clipped", round(unclipped, 3), round(clipped, 3))
metrics = [metric for _, metric in rows]
print_metric_table(rows, "clip c=1.0 accuracy")

## Results visualization

The closing figure has two parts: a small multiple showing each rung's data artifact and a summary curve of the selected metric from D1 to D5.

In [ ]:
plot_ladder_results(rungs, metrics, '6.19 Gradient clipping')

## Pitfall on D5

Clipping can hide an unsafe learning rate. On D5, pair clipping with learning-rate tuning instead of interpreting every capped update as progress.

In [ ]:
name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_for_demo(X, y)
preds_fast, losses_fast, norms_fast = train_softmax_classifier(x_tr, y_tr, x_te, epochs=40, lr=1.2, clip=1.0, seed=101)
preds_tuned, losses_tuned, norms_tuned = train_softmax_classifier(x_tr, y_tr, x_te, epochs=40, lr=0.25, clip=1.0, seed=101)
print("D5 high-lr clipped final loss:", round(losses_fast[-1], 3))
print("D5 tuned-lr clipped final loss:", round(losses_tuned[-1], 3))
print("Fix: tune lr and threshold together; inspect loss, not only clipped norm.")

## Evaluate it

- Metric: held-out accuracy from `clf_accuracy`; compare to a no-skill majority-class or plain-feature baseline.
- Sanity check: D1 should be inspectable and every probability target should sum to one when probabilities are used.
- Ablation: turn this topic's component off and verify the metric or diagnostic changes.
- Failure signal: unstable loss, axis mismatch, train/eval leakage, or D5 improvement without a matching diagnostic.

## Practice

1. Change one component value and rerun the D1 assertion plus the D1–D5 table.

2. Add a majority-class baseline to the summary curve.

3. On D5, print one extra diagnostic that would catch the named pitfall.